# Parte 6: Tarea selección abierta
Podríamos hacerla creando un modelo que ordene los posibles clientes a los que llamar según cuáles requieren de un menor tiempo de llamada hasta obtener un contrato (con un intervalo de confianza del 90% por ejemplo). Así pasamos a crear un modelo que mide regresiones. Posiblemente más útil de la empresa para analizar qué tiempo debe dedicar y crear programas de ordenación de clientes a los que llamar entre sus empleados para maximizar el número de llamadas con un alto coeficiente de éxito sin perder el tiempo de sus comerciales.




Para la métrica Brújula investigar matriz de costes.



Kernel Gaussiano para weights en KNN

Usar gráfica para entrenar Modelos

In [5]:
import pandas as pd
import numpy as np

# Cargamos el archivo del dataset de entrenamiento y obtenemos filas y columnas.
df = pd.read_pickle("banks/bank_competition.pkl")

print("1. FORMA DEL DATASET\n")
print(f"Instancias (filas): {df.shape[0]}")
print(f"Variables (columnas): {df.shape[1]}")

print("\n2. TIPO DE VARIABLES\n")
# Separamos variables numéricas de las categóricas/ordinales
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
categoricas = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Numéricas ({len(numericas)}): {numericas}")
print(f"Categóricas/Ordinales ({len(categoricas)}): {categoricas}")

print("\n3. CATEGÓRICAS CON ALTA CARDINALIDAD (>10 valores únicos)\n")
for col in categoricas:
    n_unicos = df[col].nunique()
    if n_unicos > 10:
        print(f"La variable '{col}' tiene {n_unicos} valores únicos.")

print("\n4. VALORES FALTANTES (Nulos, 'unknown' y 'other')\n")

encontrados = False
# Revisamos todas las columnas
for col in df.columns:
    # Contamos los None
    cantidad_nulos = df[col].isna().sum()

    # Contamos cuántos 'unknown' y 'other' hay (solo en texto/categoría)
    cantidad_unknown = 0
    cantidad_other = 0
    if df[col].dtype == 'object' or hasattr(df[col].dtype, 'categories'):
        cantidad_unknown = (df[col] == 'unknown').sum()
        cantidad_other = (df[col] == 'other').sum()

    # Sumamos los tres para saber el total de información faltante en esa columna
    total_faltantes = cantidad_nulos + cantidad_unknown + cantidad_other

    # Si falta algún dato, lo imprimimos con su desglose
    if total_faltantes > 0:
        porcentaje = (total_faltantes / len(df)) * 100

        # Preparamos un texto para saber exactamente qué tipo de faltantes son
        detalles = []
        if cantidad_nulos > 0:
            detalles.append(f"{cantidad_nulos} nulos (None)")
        if cantidad_unknown > 0:
            detalles.append(f"{cantidad_unknown} 'unknown'")
        if cantidad_other > 0:
            detalles.append(f"{cantidad_other} 'other'")

        print(f"La variable '{col}' tiene {total_faltantes} valores faltantes ({porcentaje:.2f}%) -> " + " y ".join(detalles))
        encontrados = True

if not encontrados:
    print("No se encontraron valores faltantes, 'unknown' ni 'other'.")

print("\n5. FILAS DUPLICADAS, COLUMNAS CONSTANTES O DE ID\n")
num_duplicados = df.duplicated().sum()
if num_duplicados > 0:
    print(f"Se han encontrado {num_duplicados} filas exactamente idénticas.")
    print("-> Se deben eliminar para evitar sesgar el modelo y reducir el tamaño del dataset.")
else:
    print("No se encontraron filas duplicadas en el dataset.")
constantes_o_id = False
for col in df.columns:
    if df[col].nunique() == 1:
        print(f"Columna constante encontrada: '{col}'")
        constantes_o_id = True
    elif df[col].nunique() == len(df):
        print(f"Columna tipo ID encontrada (todos los valores distintos): '{col}'")
        constantes_o_id = True
if not constantes_o_id:
    print("No se encontraron columnas constantes ni columnas tipo ID.")

print("\n6. TIPO DE PROBLEMA Y BALANCEO\n")
print("Es un problema de CLASIFICACIÓN (predecir 'yes' o 'no' en deposit).")
print("Distribución de la clase objetivo:")
# normalize=True nos da el porcentaje en vez del recuento
print((df['deposit'].value_counts(normalize=True) * 100).round(2))


print("\n--- VOLUMETRÍA DE LA VARIABLE 'month' ---")
# Calculamos el porcentaje de representación de cada mes en el 100% de los datos
proporciones_mes = df['month'].value_counts(normalize=True) * 100
print(proporciones_mes.round(2).astype(str) + ' %')

# Creamos una alerta para los meses con menos del 3% de los datos
meses_minoritarios = proporciones_mes[proporciones_mes < 3].index.tolist()
print(f"\n[!] ALERTA ESTRUCTURAL:")
print(f"Los siguientes meses tienen una representación marginal (< 3%): {meses_minoritarios}")
print(" -> NOTA: Al tener tan pocas filas, estas categorías sufrirán mucha varianza estadística al hacer el Train/Test Split. Habrá que vigilar posibles 'espejismos' en el EDA Profundo.")

1. FORMA DEL DATASET

Instancias (filas): 162
Variables (columnas): 16

2. TIPO DE VARIABLES

Numéricas (7): ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categóricas/Ordinales (9): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

3. CATEGÓRICAS CON ALTA CARDINALIDAD (>10 valores únicos)

La variable 'job' tiene 12 valores únicos.
La variable 'month' tiene 12 valores únicos.

4. VALORES FALTANTES (Nulos, 'unknown' y 'other')

La variable 'job' tiene 2 valores faltantes (1.23%) -> 2 'unknown'
La variable 'education' tiene 3 valores faltantes (1.85%) -> 3 'unknown'
La variable 'contact' tiene 37 valores faltantes (22.84%) -> 37 'unknown'
La variable 'poutcome' tiene 130 valores faltantes (80.25%) -> 121 'unknown' y 9 'other'

5. FILAS DUPLICADAS, COLUMNAS CONSTANTES O DE ID

No se encontraron filas duplicadas en el dataset.
No se encontraron columnas constantes ni columnas tipo ID.

6. TIPO DE PROBLEMA Y BALANCEO

Es 

KeyError: 'deposit'

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("--- ANÁLISIS DE LA INFLUENCIA DE 'DURATION' EN 'DEPOSIT' ---\n")

# 1. Filtramos los clientes de "primera llamada absoluta"
condicion = (df['previous'] == 0) & (df['campaign'] == 1)
df_primera = df[condicion].copy()

print(f"Total de clientes analizados (puerta fría pura): {len(df_primera)}\n")

# =====================================================================
# TABLA 1: Estadísticos descriptivos básicos
# =====================================================================
print("1. RESUMEN ESTADÍSTICO DE DURACIÓN (segundos):")
tabla_stats = df_primera.groupby('deposit')['duration'].describe()[['count', 'mean', '50%', 'min', 'max']]
tabla_stats.columns = ['Volumen', 'Media (s)', 'Mediana (s)', 'Mínimo (s)', 'Máximo (s)']
display(tabla_stats.round(1))

# =====================================================================
# TABLA 2: Tasa de conversión agrupada por tramos de tiempo
# =====================================================================
# Cortamos la duración en intervalos lógicos: <1 min, 1-3 min, 3-5 min, 5-10 min, +10 min
bins = [0, 60, 180, 300, 600, float('inf')]
labels = ['< 1 min', '1 - 3 min', '3 - 5 min', '5 - 10 min', '> 10 min']
df_primera['rango_tiempo'] = pd.cut(df_primera['duration'], bins=bins, labels=labels, right=False)

# Estandarizamos la columna 'deposit' a numérico (0 y 1) para poder sumar los éxitos
if df_primera['deposit'].dtype == 'object':
    df_primera['deposit_num'] = (df_primera['deposit'] == 'yes').astype(int)
else:
    df_primera['deposit_num'] = df_primera['deposit']

tabla_rangos = df_primera.groupby('rango_tiempo', observed=True).agg(
    Total_Llamadas=('deposit_num', 'count'),
    Exitos_SI=('deposit_num', 'sum')
).reset_index()

# Calculamos el % de éxito por tramo
tabla_rangos['Tasa de Éxito (%)'] = (tabla_rangos['Exitos_SI'] / tabla_rangos['Total_Llamadas'] * 100).fillna(0)

print("\n2. TASA DE CONVERSIÓN POR TRAMOS DE DURACIÓN:")
display(tabla_rangos.round(2))


# =====================================================================
# GRÁFICOS: Distribución de la duración según el éxito
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.set_style("whitegrid")

# Gráfico A: Boxplot (Cajas y Bigotes)
sns.boxplot(data=df_primera, x='deposit', y='duration', palette='Set2', ax=axes[0])
axes[0].set_title('Distribución de Duración según Decisión (Boxplot)', fontweight='bold', fontsize=14)
axes[0].set_xlabel('¿Contrata el depósito?', fontsize=12)
axes[0].set_ylabel('Duración de la llamada (segundos)', fontsize=12)

# Gráfico B: Histograma con curva de densidad (KDE)
sns.histplot(data=df_primera, x='duration', hue='deposit', kde=True, palette='Set2', ax=axes[1], bins=40, alpha=0.6)
axes[1].set_title('Volumen de Llamadas por Duración', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Duración de la llamada (segundos)', fontsize=12)
axes[1].set_ylabel('Número de clientes', fontsize=12)
axes[1].set_xlim(0, 2000) # Limitamos a 2000s (33 min) para no aplastar el gráfico con outliers extremos

plt.tight_layout()
plt.show()

print("\n[!] CONCLUSIÓN VISUAL PARA NEGOCIO:")
print("Si observas la Tasa de Éxito por tramos, verás que las llamadas cortas (<3 min) tienen un éxito cercano a cero,")
print("mientras que las llamadas largas (>10 min) tienen tasas de éxito brutales. En los gráficos, la 'caja' de los que")
print("SÍ contratan está desplazada masivamente hacia arriba.")
print("Esto justifica matemáticamente la eliminación de la columna 'duration' para evitar el Data Leakage.")

--- ANÁLISIS DE LA INFLUENCIA DE 'DURATION' EN 'DEPOSIT' ---

Total de clientes analizados (puerta fría pura): 54

1. RESUMEN ESTADÍSTICO DE DURACIÓN (segundos):


KeyError: 'deposit'

In [ ]:
# %%
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
import optuna

print("🧪 EXPERIMENTO DE FUERZA BRUTA: PIPELINE MINIMALISTA (ROC-AUC)\n" + "-"*75)

# 1. Recuperamos tus listas de variables originales (SIN 'duration')
num_cols = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

# 2. Preprocesador Básico: Cero inteligencia humana, 100% fuerza bruta matemática
preprocessor_crudo = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()) # Escalado estándar normal, sin RobustScaler ni Logaritmos
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        # handle_unknown='ignore' para evitar problemas, sin dropear nada
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
    ]), cat_cols)
])

# 3. Transformamos los datos (Asegúrate de que X_train_original no tiene 'duration')
print("[!] Preprocesando datos en bruto...")
X_train_crudo = preprocessor_crudo.fit_transform(X_train)

# 4. Optuna rápido optimizando ROC-AUC puro
def objective_raw(trial):
    c_val = trial.suggest_float('C', 0.1, 100.0, log=True)
    gamma_val = trial.suggest_float('gamma', 0.001, 1.0, log=True)
    
    model = SVC(C=c_val, gamma=gamma_val, kernel='rbf', probability=True, random_state=42, cache_size=1000)
    
    # ⚠️ OPTIMIZAMOS F2-Score AHORA
    return cross_val_score(model, X_train_crudo, y_train, cv=5, scoring=f2_scorer, n_jobs=-1).mean()

print("[!] Lanzando Optuna (F2-Score) sobre datos crudos...")
study_raw = optuna.create_study(direction='maximize')
study_raw.optimize(objective_raw, n_trials=20) # 20 intentos rápidos

print("\n" + "="*50)
print(f"🔥 F2-Score MÁXIMO (DATOS CRUDOS): {study_raw.best_value:.4f}")
print("="*50)